# Hydrometeorology data-quality EDA
Where do missing values, ranges, distributions and temporal coverage stand in the inputs we will feed a flood-prediction model? This notebook profiles the hydrometeorological sources in the same spirit as `data_quality/eda_quality_flood/data_eda.ipynb` does for the flood masks, but at the level of each forecasting *source*:
- **ERA5 precipitation and runoff**, spatially averaged over the whole Nile-basin grid and over
  the Aweil study counties.
- **AgERA5 reference evapotranspiration** at the processed grid cell.
- **Dartmouth river discharge**, one view per loaded station.
- **Lake water levels** (Victoria, Kyoga, Albert).

The goal is to choose an appropriate machine-learning model, so besides the flood-style column profile we report feature-distribution extras (zeros %, negatives %, skew) and a **supervised-set readiness** table: how many calendar days provide every feature and a flood label together.

## Run the setup below with the project environment
The first code cell finds the repository root, reloads the helper modules, runs the offline sanity checks (small made-up files, no raw data needed) and then loads the real data. The default runs one year (2024) so this cell finishes quickly. For the full record used to size the ML training set, set `YEARS = np.arange(2000, 2026)` (loads ~26 years of hourly ERA5, takes a few minutes).

In [ ]:
import importlib
import os
import sys
from contextlib import redirect_stdout
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

folders = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in folders if (p / 'raw_data').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f'Cannot find the project root (raw_data/) from {Path.cwd()}.')

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import data_quality.eda_quality_hydrometeorology.hydrometeorology_quality as q
from processing_data.loading import (
    load_dartmouth_data,
    load_lake_stations,
    load_processed_ET,
    load_rainfall_runoff,
    process_ET,
)

q = importlib.reload(q)
from data_quality.eda_quality_hydrometeorology.check_hydrometeorology_quality import (
    run_checks,
)
from data_quality.eda_quality_hydrometeorology.hydrometeorology_quality import (
    ET_LAT,
    ET_LON,
    build_source_frames,
    coverage_summary,
    distribution_extras,
    flood_label_dates,
    profile_frames,
    quality_table,
    supervised_set_readiness,
)
from EDA_hydrometeorology.hydrological_analysis import load_aweil_counties

run_checks()

OUTPUT_TABLES = PROJECT_ROOT / 'data_quality' / 'eda_quality_hydrometeorology' / 'outputs' / 'tables'
OUTPUT_FIGURES = PROJECT_ROOT / 'data_quality' / 'eda_quality_hydrometeorology' / 'outputs' / 'figures'
OUTPUT_TABLES.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES.mkdir(parents=True, exist_ok=True)

# Quick default (one year) so this cell runs fast.
# YEARS = np.arange(2000, 2026) (loads ~26 years, takes a few minutes).
YEARS = np.array([2024])
COUNTY_PATH = PROJECT_ROOT / 'raw_data' / 'Administrative boundaries' / 'ssd_admin2.geojson'
FLOOD_ROOT = PROJECT_ROOT / 'raw_data' / 'flood_masks'

counties = load_aweil_counties(COUNTY_PATH)
county_bounds = tuple(counties.total_bounds)   # (min_lon, min_lat, max_lon, max_lat)

with redirect_stdout(StringIO()):
    rainfall = load_rainfall_runoff(YEARS)
    for year in YEARS:
        process_ET(int(year), target_longitude=ET_LON, target_latitude=ET_LAT)
    et_data = load_processed_ET(YEARS, target_longitude=ET_LON, target_latitude=ET_LAT)
    dartmouth = load_dartmouth_data()
    lakes = load_lake_stations()

frames = build_source_frames(rainfall, et_data, dartmouth, lakes, county_bounds=county_bounds)
profile = profile_frames(frames)
extras = distribution_extras(frames)
coverage = coverage_summary(frames)
quality = quality_table(frames)

print(f'{len(frames)} hydrometeorological source levels profiled.')
print(f'Record window: {coverage["first_date"].min()} to {coverage["last_date"].max()} '
      f'({coverage["window_days"].iloc[0]:,} calendar days).')
print(f'Source inputs: 26 ERA5 annual .nc + 26 AgERA5 processed CSVs + '
      f'{len(dartmouth)} Dartmouth sites + 3 lake series.')

## 1. Source overview: temporal coverage and gaps
`coverage` reports, for every source, how many records and distinct dates are present over the shared observed window, the share of window days covered and the gap structure. The irregular sources (Dartmouth discharge and lake altimetry) are the interesting rows: their long gaps show where interpolation or imputation will be required.

In [ ]:
display(coverage)
print('Coverage, gaps and null-value rows per hydrometeorological source.')

## 2. Column-by-column profile
`profile` has one row per (source, column) with non-null count, NaN count and percentage, cardinality and the min/max/mean/std for value columns. `date` rows report the observed range.

In [ ]:
display(profile)
summary = pd.merge(profile, extras, on=['level', 'column'], how='left')
display(summary)
total_nan = int(profile['nan_count'].sum())
print(f'Total null/NaN values across all sources: {total_nan:,}')

## 3. Feature distributions and per-series quality (ML preprocessing)
`extras` adds % zeros, % negatives and skew for every value column - the facts that drive feature transforms (log / indicator) and model choice (scale-invariant trees vs linear/neural). `quality` reports duplicates, missing values, negatives, zeros and expected-date coverage per source.

In [ ]:
display(extras)
display(quality)

## 4. Supervised-set readiness (data readiness for model choice)
`readiness` counts, per year and in total, the calendar days that provide every feature column and (when supplied) a flood-label day. The total `days_all_features_with_label` is the size of the clean training set available; a low share relative to `days` means many rows need imputation or gap-tolerant tree models.

In [ ]:
with redirect_stdout(StringIO()):
    label_dates = flood_label_dates(FLOOD_ROOT, counties, years=YEARS)
readiness = supervised_set_readiness(frames, label_dates=label_dates)
display(readiness)
best = readiness[readiness['year'] == 'all'].iloc[0]
print(f"{best['days_all_features_with_label']:,} supervised days with all features and a "
      f"detected flood label, out of {best['days']:,} calendar days.")
print(f"{best['days_all_features']:,} days have complete features (no imputation needed).")

## 5. Figures
Three figures summarise the EDA: missing values per source and column, value ranges, and the share of the shared window each source covers.

In [ ]:
sns.set_theme(style='whitegrid', context='notebook', font_scale=1.0)

# 1) Missing values per source and column.
fig, ax = plt.subplots(figsize=(12, 5), layout='constrained')
data = profile[profile['column'] != 'date']
sns.barplot(data=data, x='level', y='nan_count', hue='column', ax=ax)
ax.tick_params(axis='x', rotation=75)
ax.set_title('Missing (null/NaN) values by source and column')
ax.set_ylabel('Missing values')
fig.savefig(OUTPUT_FIGURES / 'hydrometeorology_missing_values.png', dpi=200, bbox_inches='tight')
plt.show()

# 2) Value range (max - min) per source.
fig, ax = plt.subplots(figsize=(12, 6), layout='constrained')
value_rows = profile[(profile['column'] != 'date') & (profile['max'].notna())].copy()
value_rows['span'] = pd.to_numeric(value_rows['max'], errors='coerce') - pd.to_numeric(value_rows['min'], errors='coerce')
sns.barplot(data=value_rows, x='level', y='span', hue='column', ax=ax)
ax.tick_params(axis='x', rotation=75)
ax.set_title('Value range (max - min) by source and column')
ax.set_ylabel('Range')
fig.savefig(OUTPUT_FIGURES / 'hydrometeorology_value_ranges.png', dpi=200, bbox_inches='tight')
plt.show()

# 3) Share of the shared window with a present value.
fig, ax = plt.subplots(figsize=(12, 5), layout='constrained')
sns.barplot(data=coverage, x='level', y='coverage_pct', ax=ax, color='#3182bd')
ax.axhline(100, color='#cb181d', ls='--', lw=1)
ax.set(ylim=(0, 105))
ax.tick_params(axis='x', rotation=75)
ax.set_title('Share of the shared record window with a present value')
ax.set_ylabel('% of window days present')
fig.savefig(OUTPUT_FIGURES / 'hydrometeorology_coverage.png', dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved figures to {OUTPUT_FIGURES}')

## 6. Saving the results
The profile, distribution extras, coverage, quality report and supervised-set readiness are written under `data_quality/eda_quality_hydrometeorology/outputs/tables/`; the figures under `data_quality/eda_quality_hydrometeorology/outputs/figures/`.

In [ ]:
profile.to_csv(OUTPUT_TABLES / 'hydrometeorology_column_profile.csv', index=False)
extras.to_csv(OUTPUT_TABLES / 'hydrometeorology_distribution.csv', index=False)
coverage.to_csv(OUTPUT_TABLES / 'hydrometeorology_coverage.csv', index=False)
quality.to_csv(OUTPUT_TABLES / 'hydrometeorology_quality_report.csv', index=False)
readiness.to_csv(OUTPUT_TABLES / 'hydrometeorology_supervised_set.csv', index=False)
print(f'Saved tables to {OUTPUT_TABLES}')
for path in sorted(OUTPUT_TABLES.iterdir()) + sorted(OUTPUT_FIGURES.iterdir()):
    print(' -', path.name)

## Limitations
- ERA5 and AgERA5 are reanalysis products with a complete daily grid; their apparent 100 % coverage reflects that, not ground-truth quality. The real gaps live in the station (Dartmouth) and altimetry (lake) series.
- Dartmouth discharge and lake levels are neither daily nor complete; their `coverage_pct` and long gaps are where feature engineering (resampling, interpolation, event counts) must happen before supervised modelling.
- The Aweil ERA5 levels average over the counties' bounding box rather than a clipped polygon; they are features for modelling, not a basin closure.
- A *missing* flood label simply means that day had no detected flood pixels in the three-day composite masks; absence of a record is its own kind of missingness and is not proof the land was dry.